# Spectral clustering of time-evolving networks

This notebook implements Algorithm 1 from **Blaskovic, Conrad, Klus, and Djurdjevac Conrad, _Spectral Clustering of Time-Evolving Networks Using Spatio-Temporal Random Walks_**. It constructs the spatio-temporal random walk, removes purely temporal eigenvectors, visualizes the spatial eigenvectors and Appendix D diagnostics, clusters user-selected eigenvectors, and evaluates the result with ARI when ground truth is available.

## 1. Imports and configuration

Edit the values below and then use **Run All**. `EXAMPLE_NAME` may be one of the four bundled directories, or set `NETWORK_DIR` to your own folder. The paper uses squared temporal distance in $w_{ts}=\exp(-\alpha |t-s|^2)$. For Example 1, cyclic coupling uses $\min(|t-s|,M-|t-s|)$. Eigenvector choices are 1-based, as in the paper.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from spatiotemporal_clustering import (
    load_temporal_network, load_ground_truth, compute_spatial_eigenvectors,
    singular_vector_heuristics, cluster_embedding, adjusted_rand,
)

# ----- User configuration -----
EXAMPLE_NAME = 'example_1'  # example_1, example_2, example_3, guiding_example
NETWORK_DIR = Path('PAPER_EXAMPLES') / EXAMPLE_NAME / 'network'
GROUND_TRUTH_DIR = NETWORK_DIR.parent  # set to None if labels are unavailable
ALPHA = 0.01
CYCLIC_COUPLING = True
COUPLING_BANDWIDTH = None  # None couples all distinct snapshots; integer r couples |t-s| <= r
CUSTOM_COUPLING_MATRIX = None  # None = paper coupling; or supply/load an M x M NumPy array
N_EIGENVECTORS = 8
INITIAL_DISTRIBUTION = None  # None = uniform; or provide a nonnegative length-N array
ROW_NORMALIZE_EMBEDDING = False
RANDOM_STATE = 0

## 2. Load and validate snapshots

Adjacency matrices files should be in one of the following formats: `.csv`, `.npy` or `.npz`. Each file is one nonnegative, weighted adjacency matrix. Files are sorted by the number in their name, so, e.g. `adj_10.csv` follows `adj_9.csv`. All snapshots must be square, have equal size, and retain a consistent node ordering. An isolated node is given a self-loop when the random-walk matrices are formed.

In [ ]:
adjacencies, snapshot_files = load_temporal_network(NETWORK_DIR)
M, N = len(adjacencies), adjacencies[0].shape[0]
ground_truth = load_ground_truth(GROUND_TRUTH_DIR, (M, N)) if GROUND_TRUTH_DIR is not None else None
print(f'Loaded {M} snapshots with {N} nodes each from {NETWORK_DIR.resolve()}')
print('First/last file:', snapshot_files[0].name, '/', snapshot_files[-1].name)
print('Ground truth:', 'available' if ground_truth is not None else 'not found (ARI will be skipped)')

In [ ]:
# Optional: plot all loaded adjacency matrices.
PLOT_ADJACENCY_MATRICES = True

if PLOT_ADJACENCY_MATRICES:
    ncols = min(4, M)
    nrows = int(np.ceil(M / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.5 * nrows), squeeze=False)
    for t, ax in enumerate(axes.flat):
        if t >= M:
            ax.axis('off')
            continue
        ax.imshow(adjacencies[t].toarray(), cmap='Greys', interpolation='nearest', aspect='equal')
        ax.set_title(f'Snapshot {t + 1} ({snapshot_files[t].name})')
        ax.set_xlabel('node')
        ax.set_ylabel('node')
    plt.tight_layout()
    plt.show()

## 3. Build the spatio-temporal operator and solve its spatial spectrum

For each snapshot, equation (1) gives $S_t=D_t^{-1}A_t$. Starting from a uniform $\mu_1$, the distributions evolve as $\mu_{t+1}^\top=\mu_t^\top S_t$. Coupling weights are row-normalized to obtain $H$. The code applies the Koopman blocks $K_{ts}=S_t\cdots S_{s-1}$ and reweighted Perron–Frobenius blocks $T_{ts}=D_{\mu_s}^{-1}(S_t\cdots S_{s-1})^\top D_{\mu_t}$ from equation (11).

The snapshot-averaging projection from equation (14) removes all vectors constant within snapshots. We solve the equivalent symmetric, reversible eigenproblem, then transform its eigenvectors back to the observable coordinates used in the paper.

Spatial eigenvectors are plotted directly from `result.eigenvectors`, with no post-solve norm division, rescaling, or sign fixing. The eigensolver has an intrinsic normalization convention. Transforming symmetric coordinates back to observables (and lifting reduced coefficients) is required by the eigenproblem and is retained. Snapshot-basis orthonormalization is part of constructing the reduced operator, not an extra normalization of plotted spatial eigenvectors.

In [ ]:
result = compute_spatial_eigenvectors(
    adjacencies, alpha=ALPHA, cyclic=CYCLIC_COUPLING,
    n_eigenvectors=N_EIGENVECTORS,
    bandwidth=COUPLING_BANDWIDTH, random_state=RANDOM_STATE,
    initial_distribution=INITIAL_DISTRIBUTION,
    custom_coupling=CUSTOM_COUPLING_MATRIX,
)
print('Leading spatial eigenvalues:')
print(np.array2string(result.eigenvalues, precision=6))

## 4. Coupling matrix and its temporal modes

The heatmap shows the user-controlled coupling. The spectrum and eigenvectors of $H$ help identify slowly decaying temporal modulations that can reappear in spatial eigenvectors (Remark 3.10 and Appendix D).

In [ ]:
# H is reversible; diagonal similarity gives a symmetric matrix with the same eigenvalues.
sqrt_pi = np.sqrt(result.stationary)
H_symmetric = sqrt_pi[:, None] * result.coupling / sqrt_pi[None, :]
h_values, h_vectors_sym = np.linalg.eigh(H_symmetric)
order = np.argsort(h_values)[::-1]
h_values = h_values[order]
h_vectors = h_vectors_sym[:, order] / sqrt_pi[:, None]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
im = axes[0].imshow(result.coupling, cmap='Blues', aspect='auto')
coupling_title = ('Coupling H (custom)' if CUSTOM_COUPLING_MATRIX is not None
                  else f'Coupling H (alpha={ALPHA:g}, cyclic={CYCLIC_COUPLING})')
axes[0].set(title=coupling_title, xlabel='target snapshot', ylabel='source snapshot')
fig.colorbar(im, ax=axes[0], fraction=.046)
show_h = min(10, M)
axes[1].plot(np.arange(1, show_h + 1), h_values[:show_h], 'o-')
axes[1].set(title='Leading eigenvalues of H', xlabel='index', ylabel='eigenvalue', xticks=np.arange(1, show_h + 1))
for j in range(min(4, M)):
    axes[2].plot(h_vectors[:, j], marker='.', label=f'H mode {j+1}')
axes[2].set(title='Dominant temporal modes', xlabel='snapshot', ylabel='value')
axes[2].legend()
plt.tight_layout()

## 5. Dominant spatial eigenvalues and eigenvectors

The eigenvalues are shown as a scatterplot. As in Figures 4, 6, and 7 of the paper, every spatial eigenvector is folded into its $M$ snapshot segments and each segment is plotted as a line over the nodes. Line color progresses from dark blue for early snapshots to dark red for late snapshots. Persistent sign/value structure indicates coherent communities; localized structure can reveal splits or mergers during only part of the evolution.

In [ ]:
k_show = result.eigenvectors.shape[2]
plt.figure(figsize=(7, 4))
plt.scatter(np.arange(1, k_show + 1), result.eigenvalues, s=55, color='#2166ac')
plt.axhline(0, color='0.7', linewidth=0.8)
plt.xticks(np.arange(1, k_show + 1))
plt.xlabel('Spatial eigenvalue index')
plt.ylabel('Eigenvalue')
plt.title('Dominant spatial eigenvalues')
plt.grid(alpha=0.2)
plt.tight_layout()

ncols = min(3, k_show); nrows = int(np.ceil(k_show / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.2*nrows), squeeze=False)
snapshot_colors = plt.cm.coolwarm(np.linspace(0, 1, M))
for k, ax in enumerate(axes.flat):
    if k >= k_show:
        ax.axis('off'); continue
    for t in range(M):
        ax.plot(np.arange(1, N + 1), result.eigenvectors[t, :, k],
                color=snapshot_colors[t], linewidth=1.0, alpha=0.8)
    ax.axhline(0, color='0.75', linewidth=0.7)
    ax.set(title=f'Spatial {k+1}: lambda={result.eigenvalues[k]:.4f}', xlabel='Nodes')
    ax.margins(x=0)
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(1, M))
sm.set_array([])
fig.subplots_adjust(wspace=.28, hspace=.42, right=.88)
colorbar_axis = fig.add_axes([0.91, 0.15, 0.015, 0.70])
fig.colorbar(sm, cax=colorbar_axis, label='Snapshots')

## 6. Appendix D singular-vector heuristics

For spatial eigenvector $k$, arrange its snapshot segments as rows of $F_k\in\mathbb{R}^{M\times N}$ and compute $F_k=\Phi_k\Sigma_k\Psi_k^\top$. A large $\varsigma_1/\varsigma_2$ means the eigenvector is approximately rank one. Compare its dominant left singular vector $\phi_1^k$ with non-constant modes of $H$: a close match suggests a repeated spatial observable modulated by coupling geometry and may be redundant for clustering. The right singular vector $\psi_1^k$ shows that base spatial observable. These are diagnostics, not an automatic selection rule.

In [ ]:
diagnostics = singular_vector_heuristics(result.eigenvectors)
fig, axes = plt.subplots(3, 1, figsize=(9, 9))
for k, diagnostic in enumerate(diagnostics):
    sv = diagnostic['singular_values']
    axes[0].semilogy(np.arange(1, min(8, len(sv)) + 1), sv[:8], 'o-', label=f'spatial {k+1}')
    left = diagnostic['left_vector'].copy()
    if left[np.argmax(np.abs(left))] < 0: left *= -1
    axes[1].plot(left, label=f'Left sing. vector {k+1}')
    
    axes[2].plot(diagnostic['right_vector'], label=f'Right sing. vector {k+1}')
axes[0].set(title='Singular-value decay of each F_k', xlabel='singular-value index', ylabel='singular value')
axes[1].set(title='Dominant left singular vectors (temporal modulation)', xlabel='snapshot', ylabel='value')
axes[2].set(title='Dominant right singular vectors (base spatial observables)', xlabel='node', ylabel='value')
for ax in axes: ax.legend(ncol=min(4, k_show), fontsize=8)
plt.tight_layout()

## 7. Cluster selected spatial eigenvectors

Manually set `SELECTED_EIGENVECTORS` and `N_CLUSTERS` at the top of the next cell after inspecting the plots. Eigenvector indices are 1-based, matching the plot titles and the paper. K-means is applied to one row per space-time node. The resulting label numbers are arbitrary; ARI correctly ignores label permutations.

In [ ]:
# ----- Manual clustering choices -----
SELECTED_EIGENVECTORS = [1, 3, 4]  # 1-based indices; choose any plotted spatial eigenvectors
N_CLUSTERS = 6                  # requested number of k-means communities

labels, embedding = cluster_embedding(
    result.eigenvectors, SELECTED_EIGENVECTORS, N_CLUSTERS,
    random_state=RANDOM_STATE, row_normalize=ROW_NORMALIZE_EMBEDDING,
)
ari = adjusted_rand(labels, ground_truth)
print(f'Used spatial eigenvectors {SELECTED_EIGENVECTORS}; clusters={N_CLUSTERS}')
print(f'Adjusted Rand index: {ari:.6f}' if ari is not None else 'Adjusted Rand index: skipped (no ground truth)')

fig, axes = plt.subplots(1, 2 if ground_truth is not None else 1, figsize=(14 if ground_truth is not None else 7, 4), squeeze=False)
im = axes[0, 0].imshow(labels.T, cmap='jet', interpolation='nearest', aspect='auto')
axes[0, 0].set(title='Predicted space-time communities', xlabel='Snapshots', ylabel='Nodes')
if ground_truth is not None:
    im = axes[0, 1].imshow(ground_truth.T, cmap='jet', interpolation='nearest', aspect='auto')
    axes[0, 1].set(title=f'Ground truth (ARI={ari:.4f})', xlabel='Snapshots', ylabel='Nodes')
plt.tight_layout()

## 8. [OPTIONAL] 2D-embedding view
If at least two eigenvectors were selected, this plot shows the first two feature coordinates. Uncomment the final lines to save labels in the same `(snapshots, nodes)` layout as the ground truth.

In [ ]:
if embedding.shape[1] >= 2:
    plt.figure(figsize=(7, 6))
    plt.scatter(embedding[:, 0], embedding[:, 1], c=labels.ravel(), cmap='tab20', s=8, alpha=.7)
    plt.xlabel(f'spatial eigenvector {SELECTED_EIGENVECTORS[0]}')
    plt.ylabel(f'spatial eigenvector {SELECTED_EIGENVECTORS[1]}')
    plt.title('Space-time node embedding colored by predicted community')
    plt.show()
else:
    print('Select at least two eigenvectors for a 2-D embedding plot.')

# np.save(f'{EXAMPLE_NAME}_predicted_labels.npy', labels)
# np.savetxt(f'{EXAMPLE_NAME}_predicted_labels.csv', labels, fmt='%d', delimiter=',')